In [ ]:
import json, pathlib

inventory = json.loads(
    pathlib.Path(
        "/scr/BEHAVIOR-1K/asset_pipeline/artifacts/pipeline/object_inventory.json"
    ).read_text()
)
providers = {k.split("-")[1]: v for k, v in inventory["providers"].items()}

In [ ]:
import sys

sys.path.append("/scr/BEHAVIOR-1K/asset_pipeline")
from b1k_pipeline.utils import parse_name, load_mesh

In [ ]:
EXCLUDE_CATEGORIES = {
    "walls",
    "floors",
    "ceilings",
    "driveway",
    "lawn",
    "background",
    "roof",
}

In [ ]:
import csv

# Load the rename file
RENAMES = {}
with open("/scr/BEHAVIOR-1K/asset_pipeline/metadata/object_renames.csv") as f:
    for row in csv.DictReader(f):
        key = row["ID (auto)"]
        RENAMES[key] = row["New Category"]

# Load the deletion file
DELETION_QUEUE = set()
with open("/scr/BEHAVIOR-1K/asset_pipeline/metadata/deletion_queue.csv", "r") as f:
    for row in csv.DictReader(f):
        DELETION_QUEUE.add(row["Object"].strip().split("-")[1])


def maybe_rename_category(cat, model):
    if model in RENAMES:
        return RENAMES[model]
    return cat

In [ ]:
# Process the collision mesh directory
meshdir = pathlib.Path("/scr/cmesh_dataset")

# For every directory in there that is named after a parseable object, rename it
for item in meshdir.glob("*"):
    if item.is_dir():
        name = item.name
        if name.count("-") == 2:
            continue

        parsed_name = parse_name(name)
        if not parsed_name:
            print("Bad name", name)
            continue

        link_name = (
            parsed_name.group("link_name")
            if parsed_name.group("link_name")
            else "base_link"
        )
        obj_name = (
            parsed_name.group("category")
            + "-"
            + parsed_name.group("model_id")
            + "-"
            + link_name
        )
        new_path = meshdir / obj_name
        if new_path.exists():
            print("Collision", new_path, item)
            continue
        item.rename(new_path)

In [ ]:
import shutil

# Remove objects that are substances. Those collision meshes will not make sense
from bddl.object_taxonomy import ObjectTaxonomy

ot = ObjectTaxonomy()

for item in meshdir.glob("*"):
    if item.is_dir():
        name = item.stem
        category, mdl, link = name.split("-")
        assert len(mdl) == 6, mdl

        if mdl in DELETION_QUEUE:
            print("Deleting deletion queue item", item)
            shutil.rmtree(item)
            continue

        renamed_category = maybe_rename_category(category, mdl)
        is_substance = ot.get_synset_from_substance(renamed_category) is not None
        is_object = ot.get_synset_from_category(renamed_category) is not None
        assert is_substance or is_object, (renamed_category, item)
        if is_substance:
            print("Deleting substance", item)
            shutil.rmtree(item)
            continue

        if renamed_category in EXCLUDE_CATEGORIES:
            print("Deleting excluded category", item)
            shutil.rmtree(item)
            continue

        # Apply the rename if necessary
        if category != renamed_category:
            new_path = meshdir / (renamed_category + "-" + mdl + "-" + link)
            if new_path.exists():
                print("Collision", new_path, item)
                continue
            item.rename(new_path)

In [ ]:
# Now load the JSONs
selections = {}
for fn in pathlib.Path("/scr/BEHAVIOR-1K/asset_pipeline/cad").glob(
    "*/*/artifacts/collision_selection.json"
):
    target = "/".join(fn.parts[-4:-2])
    for name, selection in json.loads(fn.read_text()).items():
        parsed_name = parse_name(name)
        if not parsed_name:
            print("Bad name", name)
            continue

        # model_id = parsed_name.group("model_id")
        # if model_id in DELETION_QUEUE or model_id not in providers:
        #     print("Deleting deletion queue item", name)
        #     continue

        if providers[parsed_name.group("model_id")] != target:
            print("Mismatch", name, target)
            continue

        # renamed_category = maybe_rename_category(parsed_name.group("category"), model_id)
        # is_substance = ot.get_synset_from_substance(renamed_category) is not None
        # if is_substance:
        #     print("Deleting substance", name)
        #     continue

        # if renamed_category in EXCLUDE_CATEGORIES:
        #     print("Deleting excluded category", name)
        #     continue

        link_name = (
            parsed_name.group("link_name")
            if parsed_name.group("link_name")
            else "base_link"
        )
        selection_name = parsed_name.group("model_id") + "-" + link_name

        selections[selection_name] = selection

In [ ]:
from fs.zipfs import ZipFS
import fs.path

for fn in pathlib.Path("/scr/BEHAVIOR-1K/asset_pipeline/cad").glob(
    "*/*/artifacts/meshes.zip"
):
    target = "/".join(fn.parts[-4:-2])
    with ZipFS(fn) as zip_fs:
        for name in zip_fs.glob("*/*Mcollision*.obj"):
            name = fs.path.splitext(fs.path.basename(name.path))[0].rsplit("-", 1)[0]
            parsed_name = parse_name(name)
            if not parsed_name:
                print("Bad name", name)
                continue

            if providers[parsed_name.group("model_id")] != target:
                print("Mismatch", name, target)
                continue

            model_id = parsed_name.group("model_id")
            renamed_category = maybe_rename_category(
                parsed_name.group("category"), model_id
            )
            link_name = (
                parsed_name.group("link_name")
                if parsed_name.group("link_name")
                else "base_link"
            )
            selection_name = parsed_name.group("model_id") + "-" + link_name
            selections[selection_name] = "manual"

In [ ]:
selections

In [ ]:
all_in_dataset = [x.name for x in meshdir.glob("*") if x.is_dir()]
full_name_selections = {}
for name in all_in_dataset:
    selection_key = name.split("-", 1)[1]
    if selection_key not in selections:
        print("No selection for", name)
        continue
    full_name_selections[name] = selections[selection_key]

In [ ]:
# Check that the keys in the dataset and the JSON are the same
in_json = set(full_name_selections.keys())
in_dataset = set([x.name for x in meshdir.glob("*") if x.is_dir()])
print("In JSON but not in dataset:", in_json - in_dataset)
print("In dataset but not in JSON:", in_dataset - in_json)
assert in_json == in_dataset

In [ ]:
# Dump the JSON file
with open("/scr/cmesh_dataset/collision_selection.json", "w") as f:
    json.dump(selections, f, indent=4)

In [ ]:
# Some quick analysis: plot the selection counts
import matplotlib.pyplot as plt
from collections import Counter

c = Counter(full_name_selections.values())
plt.bar(c.keys(), c.values())
plt.xticks(rotation=90)
plt.show()

In [ ]:
# How many candidates does each object have?
n_candidates = {}
for item in meshdir.glob("*"):
    if not item.is_dir():
        continue

    name = item.name
    candidates = {x.stem.rsplit("-", 1)[0] for x in item.glob("*.obj")}
    print(candidates)
    n_candidates[name] = len(candidates)

plt.hist(n_candidates.values(), bins=range(0, 10))
plt.show()

In [ ]:
# Create combined meshes
for item in meshdir.glob("*"):
    if not item.is_dir():
        continue

    name = item.name
    candidates = {x.stem.rsplit("-", 1)[0] for x in item.glob("*.obj")}